# 0. Imports

In [94]:
import re
from collections import Counter, defaultdict
from pathlib import Path

import faiss
import networkx as nx
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from gensim.models import Word2Vec
from node2vec import Node2Vec
from torch.utils.data import DataLoader, Dataset

# 1. Download and Store Dataset

In [95]:
def ingest_reddit_data(
    subreddit_key: str, n_rows: int = 1_000_000, force_rerun: bool = False
) -> Path:
    """
    Orchestrates the ETL process for a specific subreddit's comment data.

    Args:
        subreddit_key: Dictionary key from 'splits' (e.g., 'changemyview').
        n_rows: Maximum records to process for the local sample.
        force_rerun: If True, bypasses existence check and overwrites existing parquet file.

    Returns:
        Path to the processed Parquet file.
    """
    out_path = Path(f"data/processed/{subreddit_key}_sample.parquet")

    # Idempotency check: Skip heavy network I/O if the target file is already present
    if out_path.exists() and not force_rerun:
        print(f"Skipping ingestion: Local cache found at {out_path}")
        return out_path

    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Define schema subset based on downstream analytical requirements
    feature_cols = [
        "author",
        "body",
        "created_utc",
        "id",
        "link_id",
        "name",
        "parent_id",
        "score",
        "controversiality",
        "total_awards_received",
    ]
    splits = {
        "changemyview": "data/changemyview-*-of-*.parquet",
    }

    print(f"Streaming data from HuggingFace for: r/{subreddit_key}...")

    # Execute lazy-evaluated ETL pipeline
    try:
        (
            pl.scan_parquet(
                f"hf://datasets/HuggingFaceGECLM/REDDIT_comments/{splits[subreddit_key]}"
            )
            .select(feature_cols)
            # Filter out deleted/removed content to maintain high data quality for NLP tasks
            .filter(~pl.col("body").is_in(["[deleted]", "[removed]"]))
            .limit(n_rows)
            # Stream directly to disk using ZSTD to balance compression ratio and write speed
            .sink_parquet(out_path, compression="zstd")
        )
        print(f"Successfully wrote {n_rows} rows to {out_path}")
    except KeyError:
        raise ValueError(f"Subreddit '{subreddit_key}' not found in defined splits.")
    except Exception as e:
        print(f"Pipeline failed: {e}")
        raise

    return out_path


# --- Execution Control ---
# Toggle 'force_rerun' if the upstream data schema changes or a larger sample is needed
OUT = ingest_reddit_data("changemyview", n_rows=1_000_000, force_rerun=False)

Skipping ingestion: Local cache found at data/processed/changemyview_sample.parquet


# 2. Load Data from Parquet File

In [96]:
df = (
    # Scan the metadata and define the lazy query plan
    pl.scan_parquet("data/processed/changemyview_sample.parquet")
    # Constrain sample size for rapid local prototyping
    .head(10000)
    # Trigger execution and load into memory
    .collect()
    # Bridge to Pandas for ecosystem compatibility
    .to_pandas()
)

# 3. Create Train and Test Dataset

### 3.1 Data Preprocessing, Temporal Splitting & Metadata Mapping

In [97]:
# --- 1. Data Cleaning & Type Casting ---

# Ensure text integrity by removing null observations in the primary feature
df = df.dropna(subset=["body"])

# Filter out anonymous/deleted accounts to maintain attribution quality
df = df[df["author"] != "[deleted]"]

# Normalize timestamps: Convert raw strings to numeric Unix seconds, then to datetime objects
# 'coerce' handles malformed strings by returning NaT, preventing pipeline crashes
df["created_utc"] = pd.to_numeric(df["created_utc"], errors="coerce")
df["date"] = pd.to_datetime(df["created_utc"], unit="s", errors="coerce")


# --- 2. Temporal Train/Test Split ---

# Use a temporal 80/20 split rather than a random shuffle to prevent 'look-ahead' bias.
# This simulates a real-world scenario where we predict future comments based on past data.
cutoff = df["created_utc"].quantile(0.8)

df_train = df[df["created_utc"] <= cutoff].copy()
df_test = df[df["created_utc"] > cutoff].copy()


# --- 3. Metadata Mapping (Lookup Tables) ---

# Create lightweight author lookups for efficient O(1) retrieval.
# Mappings are scoped strictly within splits to enforce isolation and prevent leakage.
id2author_train = df_train.set_index("id")["author"].to_dict()
id2author_test = df_test.set_index("id")["author"].to_dict()

### 3.2 Interaction Network Construction

In [98]:
def build_reply_pairs(df_split, id2author):
    """
    Constructs a positive interaction dataset by mapping comments to their parent authors.
    Filters for comment-to-comment replies and removes self-interactions.
    """
    # Reddit 'parent_id' prefixes: t1 = Comment, t3 = Link/Post.
    # We restrict analysis to comment-to-comment interactions to capture conversational dynamics.
    parent_comment_ids = df_split["parent_id"].astype(str)
    is_comment_reply = parent_comment_ids.str.startswith("t1_")
    df_r = df_split[is_comment_reply].copy()

    # Extract the raw 36-base ID by stripping the 't1_' type prefix for join compatibility
    df_r["parent_key"] = df_r["parent_id"].str.replace("^t1_", "", regex=True)

    # Resolve parent author identities via the provided lookup table (O(1) mapping)
    df_r["parent_author"] = df_r["parent_key"].map(id2author)

    # --- Data Integrity & Quality Filtering ---
    # 1. Drop replies where the parent comment falls outside the current split (boundary integrity)
    df_r = df_r.dropna(subset=["parent_author"])
    # 2. Exclude self-replies to ensure we only model interpersonal interactions
    df_r = df_r[df_r["author"] != df_r["parent_author"]]

    # Feature selection and renaming to standard (u, v) graph notation
    pairs_pos = df_r[
        ["author", "parent_author", "created_utc", "link_id", "id", "parent_key"]
    ].copy()
    pairs_pos = pairs_pos.rename(
        columns={
            "author": "u",
            "parent_author": "v",
            "id": "u_comment_id",
            "parent_key": "v_comment_id",
        }
    )

    # Label as positive instances for downstream binary classification
    pairs_pos["y"] = 1
    return pairs_pos


# Generate interaction sets; scoped within splits to prevent data leakage
pos_train = build_reply_pairs(df_train, id2author_train)
pos_test = build_reply_pairs(df_test, id2author_test)

print(f"Positive samples - Train: {len(pos_train):,} | Test: {len(pos_test):,}")

Positive samples - Train: 4,356 | Test: 928


In [99]:
# --- Graph Diagnostics: Sparsity & Degree Distribution ---

# Calculate the ratio of users who engaged in at least one reply
pos_users = set(pos_train["u"]) | set(pos_train["v"])
all_users = set(df_train["author"].dropna().unique())
print(
    f"Engagement Coverage: {len(pos_users)} / {len(all_users)} users with interactions"
)

# Analyze the 'Out-Degree' (number of replies sent per user)
print("\nReplies per user statistics:")
print(pos_train.groupby("u").size().describe())

Engagement Coverage: 999 / 1321 users with interactions

Replies per user statistics:
count    884.000000
mean       4.927602
std        9.560404
min        1.000000
25%        1.000000
50%        2.000000
75%        5.000000
max      123.000000
dtype: float64


### 3.3 Negative Sampling Strategy

In [100]:
def build_hard_negatives(df_split, pos_pairs, k_per_pos=2, seed=42):
    """
    Generates 'hard' negative samples for link prediction by identifying potential
    interactions that did NOT occur within the same discussion thread context.
    """
    # Initialize a BitGenerator for reproducible stochastic sampling
    rng = np.random.default_rng(seed)

    # 1) Contextual Mapping: Identify all active participants per discussion thread (link_id).
    # This defines our 'closed-world' candidate pool for each observation.
    thread_users = (
        df_split.groupby("link_id")["author"].apply(lambda s: set(s.dropna())).to_dict()
    )

    # 2) Network Topology: Extract existing interaction edges in (u, v) space.
    # We treat edges as symmetric to prevent sampling reciprocal replies as negatives,
    # which would introduce label noise.
    reply_edges = set(zip(pos_pairs["u"], pos_pairs["v"]))
    reply_edges_sym = reply_edges | {(v, u) for (u, v) in reply_edges}

    neg_rows = []
    # Project to minimal feature set to reduce overhead during iteration
    pos_pairs_small = pos_pairs[["u", "v", "link_id"]].copy()

    for u, v, link_id in pos_pairs_small.itertuples(index=False):
        users = list(thread_users.get(link_id, []))
        if len(users) <= 1:
            continue

        # Candidate Filtering:
        # Target users in the same thread (high-signal 'hard' negatives) excluding the source 'u'
        cand = [x for x in users if x != u]
        if not cand:
            continue

        # Collision Avoidance: Remove candidates where a ground-truth interaction (u, x) exists
        cand = [x for x in cand if (u, x) not in reply_edges_sym]
        if not cand:
            continue

        # Stochastic Sampling: Select 'k' negatives per positive to maintain class ratio
        take = min(k_per_pos, len(cand))
        sampled = rng.choice(cand, size=take, replace=False)

        for x in sampled:
            neg_rows.append((u, x, link_id, 0))

    return pd.DataFrame(neg_rows, columns=["u", "v", "link_id", "y"])


# --- Triplet Dataset Assembly ---


def build_triplets_from_hard_negatives(pos_pairs, neg_pairs, seed=42):
    """
    Constructs triplets (u, v_pos, v_neg) for metric learning.
    For each positive interaction (u, v_pos), sample one hard negative v_neg
    from the same thread context.
    """
    rng = np.random.default_rng(seed)

    # Map u → list of negative candidates
    neg_map = neg_pairs.groupby("u")["v"].apply(list).to_dict()

    triplets = []

    for u, v_pos, link_id in pos_pairs[["u", "v", "link_id"]].itertuples(index=False):
        neg_candidates = neg_map.get(u, [])
        if not neg_candidates:
            continue

        # Sample one hard negative for this positive
        v_neg = rng.choice(neg_candidates)

        triplets.append((u, v_pos, v_neg, link_id))

    return pd.DataFrame(triplets, columns=["u", "v_pos", "v_neg", "link_id"])


# --- Generate hard negatives (unchanged) ---
neg_train = build_hard_negatives(df_train, pos_train, k_per_pos=2)
neg_test = build_hard_negatives(df_test, pos_test, k_per_pos=2)

# --- Build triplets ---
triplets_train = build_triplets_from_hard_negatives(pos_train, neg_train)
triplets_test = build_triplets_from_hard_negatives(pos_test, neg_test)

print("Triplets Train:", len(triplets_train))
print("Triplets Test:", len(triplets_test))
print(triplets_train.head())


Triplets Train: 4322
Triplets Test: 921
                 u                 v_pos            v_neg    link_id
0        Jaberkaty  Thompson_S_Sweetback  ancillarynipple  t3_16ralh
1  ancillarynipple  Thompson_S_Sweetback        Jaberkaty  t3_16ralh
2  ancillarynipple  Thompson_S_Sweetback        Jaberkaty  t3_16ralh
3           llatia             gchase723            xmreg  t3_16s6jg
4     cardswsbound                 nix0n         Rongoose  t3_16rzx1


In [101]:
def test_triplet_integrity(triplets, pos_df, neg_df):
    """
    Comprehensive suite to verify triplet logic and data leakage.
    """
    # 1. Structural Check
    assert not triplets.isnull().values.any(), "Triplets contain NaN values"

    # 2. Contextual Integrity: v_neg must actually exist in the negative pool for that user
    # This ensures rng.choice didn't pull a random user from the wrong context
    u_to_negs = neg_df.groupby("u")["v"].apply(set).to_dict()

    for row in triplets.itertuples():
        # Check: v_pos and v_neg must be different
        assert row.v_pos != row.v_neg, f"Anchor {row.u} has identical Pos/Neg target"

        # Check: anchor cannot be its own target
        assert row.u != row.v_pos, f"Self-loop found in positive: {row.u}"
        assert row.u != row.v_neg, f"Self-loop found in negative: {row.u}"

        # Check: v_neg must be a valid 'hard' negative from the pool
        valid_negs = u_to_negs.get(row.u, set())
        assert row.v_neg in valid_negs, (
            f"User {row.v_neg} is not a valid hard negative for {row.u}"
        )

    # 3. Label Leakage: Ensure v_neg is NEVER a real positive for that user
    # (Symmetric check to be extra safe)
    pos_edges = set(zip(pos_df["u"], pos_df["v"]))
    pos_edges_sym = pos_edges | {(v, u) for (u, v) in pos_edges}

    triplet_neg_edges = set(zip(triplets["u"], triplets["v_neg"]))
    overlap = triplet_neg_edges.intersection(pos_edges_sym)

    assert len(overlap) == 0, (
        f"Leakage detected! {len(overlap)} 'negatives' are actually real interactions."
    )

    print("✅ All Triplet Integrity Tests Passed!")


# Run the test
test_triplet_integrity(triplets_train, pos_train, neg_train)

✅ All Triplet Integrity Tests Passed!


### 3.4 User Textual Profile Construction

In [102]:
# --- 1. Corpus Preparation & Leakage Prevention ---

# Isolate training and testing text to ensure that future comments do not
# influence the historical representations of users in the training set.
df_train_text = df_train.dropna(subset=["body", "id"]).copy()
df_test_text = df_test.dropna(subset=["body", "id"]).copy()

# (Optional) Heuristic: Filter for active users to ensure embeddings have
# sufficient signal (min 5 observations).
# df_train_text = df_train_text.groupby("id").filter(lambda g: len(g) >= 5)

# --- 2. Temporal Aggregation (Feature Engineering) ---

# Construct a profile for each author.
# We join the most recent comments to capture the user's current interests/voice.
user_text_train = (
    df_train_text.sort_values(
        "created_utc"
    )  # Enforce chronology to correctly identify the 'tail'
    .groupby("author")["body"]
    # Hyperparameter: Concatenating the last 10 comments balances context vs. sequence length
    .apply(lambda s: " ".join(s.tail(10)))
)

user_text_test = (
    df_test_text.sort_values("created_utc")
    .groupby("author")["body"]
    .apply(lambda s: " ".join(s.tail(10)))
)

# Convert to hash maps (dict) for O(1) lookup performance during the mapping phase
user_text_dict_train = user_text_train.to_dict()
user_text_dict_test = user_text_test.to_dict()

# --- 3. Coverage Analysis (Data Integrity Check) ---

# Quantify the 'Cold-Start' issue: users in the interaction pairs who lack
# textual history. Significant missingness here indicates a sampling mismatch.
missing_train = triplets_train["u"].map(user_text_dict_train).isna().mean()
print(f"Missing text profile ratio (Train - Source User): {missing_train:.2%}")

missing_test = triplets_test["u"].map(user_text_dict_test).isna().mean()
print(f"Missing text profile ratio (Test - Source User): {missing_test:.2%}")

Missing text profile ratio (Train - Source User): 0.00%
Missing text profile ratio (Test - Source User): 0.00%


In [103]:
# --- 1. Prepare Data Containers ---

# Create copies to prevent SettingWithCopy warnings and isolate split changes
triplets_train = triplets_train.copy()
triplets_test = triplets_test.copy()


def attach_text(triplets, user_text_dict):
    """Adds historical text for source (u) and target (v) users."""

    # Map text profiles to user IDs
    triplets["text_u"] = triplets["u"].map(user_text_dict)
    triplets["text_v_pos"] = triplets["v_pos"].map(user_text_dict)
    triplets["text_v_neg"] = triplets["v_neg"].map(user_text_dict)

    # Remove observations missing text for either user to ensure a complete feature set
    return triplets.dropna(subset=["text_u", "text_v_pos", "text_v_neg"])


# --- 2. Execute Merge & Cleanup ---

triplets_train_txt = attach_text(triplets_train, user_text_dict_train)
triplets_test_txt = attach_text(triplets_test, user_text_dict_test)

# --- 3. Progress Check ---

# Log row counts to monitor data loss during the mapping/dropping process
print(f"Train Retention: {len(triplets_train):,} -> {len(triplets_train_txt):,}")
print(f"Test Retention:  {len(triplets_test):,} -> {len(triplets_test_txt):,}")

Train Retention: 4,322 -> 4,322
Test Retention:  921 -> 921


### 3.5 Save Train and Test Datasets

In [104]:
triplets_train_txt.to_parquet("data/processed/train_triplets_txt.parquet", index=False)
triplets_test_txt.to_parquet("data/processed/test_triplets_txt.parquet", index=False)

# 4. Train CNN

### 4.1 Create Vocabulary

In [105]:
# Define a simple regex to extract word-level tokens (alphabetic only)
TOKEN_RE = re.compile(r"[A-Za-z']+")


def tokenize(text: str):
    """Lowercases and extracts valid word tokens from raw text."""
    return TOKEN_RE.findall(text.lower())


# Constraints for memory efficiency and noise reduction
MAX_VOCAB = 50_000
MIN_FREQ = 2

# Build frequency distribution from training corpus only to prevent leakage
counter = Counter()
for t in triplets_train_txt["text_u"].tolist():
    counter.update(tokenize(t))
for t in triplets_train_txt["text_v_pos"].tolist():
    counter.update(tokenize(t))
for t in triplets_train_txt["text_v_neg"].tolist():
    counter.update(tokenize(t))

# Reserved tokens for sequence padding and out-of-vocabulary terms
PAD = "<pad>"
UNK = "<unk>"

# Initialize vocabulary with reserved indices
vocab = {PAD: 0, UNK: 1}

# Populate vocabulary with the most frequent terms meeting the frequency threshold
for w, c in counter.most_common(MAX_VOCAB):
    if c < MIN_FREQ:
        break
    vocab[w] = len(vocab)

pad_id = vocab[PAD]
unk_id = vocab[UNK]

print(f"Final Vocab Size: {len(vocab):,}")

Final Vocab Size: 19,328


### 4.2 Dataset Definition & User Text Encoding

In [106]:
# Fixed sequence length to ensure uniform input dimensions for the model
MAX_LEN = 256


def encode(text: str):
    """
    Converts raw text into a list of integer IDs.
    Unknown words are mapped to 'unk_id' and sequences are truncated to MAX_LEN.
    """
    ids = [vocab.get(w, unk_id) for w in tokenize(text)]
    return ids[:MAX_LEN]


class TripletDataset(Dataset):
    """
    Custom PyTorch Dataset to serve (u, v_pos, v_neg) triplets, their respective
    encoded histories, and the binary interaction label.
    """

    def __init__(self, df_triplets):
        self.u = df_triplets["u"].tolist()
        self.v_pos = df_triplets["v_pos"].tolist()
        self.v_neg = df_triplets["v_neg"].tolist()
        self.u_texts = df_triplets["text_u"].tolist()
        self.v_pos_texts = df_triplets["text_v_pos"].tolist()
        self.v_neg_texts = df_triplets["text_v_neg"].tolist()

    def __len__(self):
        return len(self.u)

    def __getitem__(self, idx):
        # Returns raw IDs and encoded text sequences for the given index
        return (
            self.u[idx],
            self.v_pos[idx],
            self.v_neg[idx],
            encode(self.u_texts[idx]),
            encode(self.v_pos_texts[idx]),
            encode(self.v_neg_texts[idx]),
        )

In [107]:
def collate_fn(batch):
    """
    Dynamic padding: Aligns sequences within a batch to the length of
    the longest sequence found in that specific batch.
    """
    # Unpack columns from the batch of tuples
    u, v_pos, v_neg, u_seqs, v_pos_seqs, v_neg_seqs = zip(*batch)

    # Track original lengths for masking or sequence packing
    u_lens = torch.tensor([len(s) for s in u_seqs], dtype=torch.long)
    v_pos_lens = torch.tensor([len(s) for s in v_pos_seqs], dtype=torch.long)
    v_neg_lens = torch.tensor([len(s) for s in v_neg_seqs], dtype=torch.long)

    # Determine batch-wide maximum dimensions
    max_u = max(u_lens).item()
    max_v_pos = max(v_pos_lens).item()
    max_v_neg = max(v_neg_lens).item()

    # Initialize tensors filled with the PAD token
    u_tensor = torch.full((len(batch), max_u), pad_id, dtype=torch.long)
    v_pos_tensor = torch.full((len(batch), max_v_pos), pad_id, dtype=torch.long)
    v_neg_tensor = torch.full((len(batch), max_v_neg), pad_id, dtype=torch.long)

    # Copy sequence data into the padded containers
    for i, s in enumerate(u_seqs):
        u_tensor[i, : len(s)] = torch.tensor(s, dtype=torch.long)

    for i, s in enumerate(v_pos_seqs):
        v_pos_tensor[i, : len(s)] = torch.tensor(s, dtype=torch.long)

    for i, s in enumerate(v_neg_seqs):
        v_neg_tensor[i, : len(s)] = torch.tensor(s, dtype=torch.long)

    return (
        list(u),
        list(v_pos),
        list(v_neg),
        u_tensor,
        v_pos_tensor,
        v_neg_tensor,
        u_lens,
        v_pos_lens,
        v_neg_lens,
    )

### 4.3 Data Loader Initilization

In [108]:
# Number of samples processed before the model updates its internal parameters
BATCH_SIZE = 128

# Instantiate dataset objects for training and evaluation
train_ds = TripletDataset(triplets_train_txt)
test_ds = TripletDataset(triplets_test_txt)

# Training Loader: Shuffle enabled to prevent the model from learning the order of samples
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)

# Testing Loader: Shuffle disabled to ensure consistent, reproducible evaluation
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

### 4.4 Create Siamese CNN

##### 4.4.1 Create Text Encoder to build user embeddings

In [109]:
class TextCNNEncoder(nn.Module):
    """
    Multi-kernel CNN for extracting hierarchical n-gram features from text.
    Outputs a normalized embedding representing a user's linguistic style.
    """

    def __init__(
        self,
        vocab_size,
        emb_dim=128,
        num_filters=128,
        kernel_sizes=(3, 4, 5),
        out_dim=128,
        pad_idx=0,
        dropout=0.2,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        # Parallel convolutional layers to capture varied phrase lengths
        self.convs = nn.ModuleList(
            [
                nn.Conv1d(in_channels=emb_dim, out_channels=num_filters, kernel_size=k)
                for k in kernel_sizes
            ]
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), out_dim)

    def forward(self, x):
        # x: [Batch, Sequence_Length]
        emb = self.embedding(x)
        emb = emb.transpose(1, 2)  # Align dimensions for 1D convolution [B, E, T]

        conv_outs = []
        for conv in self.convs:
            # Apply convolution and non-linearity
            h = F.relu(conv(emb))
            # Global Max Pooling: Extract the most salient feature per filter
            h = F.max_pool1d(h, kernel_size=h.size(2)).squeeze(2)
            conv_outs.append(h)

        # Fusion of multi-scale features
        h = torch.cat(conv_outs, dim=1)
        h = self.dropout(h)
        h = self.fc(h)

        # L2 Normalization to facilitate cosine similarity downstream
        h = F.normalize(h, p=2, dim=1)
        return h

##### 4.4.2 Create Siamese Architecture

In [110]:
class SiameseCNN(nn.Module):
    """
    Siamese architecture for metric learning.
    Uses a shared encoder to map the anchor, positive, and negative
    samples into a common vector space.
    """

    def __init__(self, encoder: nn.Module):
        super().__init__()
        # The core Siamese principle: one encoder, shared weights.
        self.encoder = encoder

    def forward(self, u_tensor, v_pos_tensor, v_neg_tensor):
        # Pass all three through the identical encoder
        # This maps them to the same latent space for distance comparison
        emb_u = self.encoder(u_tensor)
        emb_pos = self.encoder(v_pos_tensor)
        emb_neg = self.encoder(v_neg_tensor)

        return emb_u, emb_pos, emb_neg

##### 4.4.3 Model Instatntiation & Device Allocation

In [111]:
# Automatically detect if a GPU is available for accelerated training
device = torch.device("mps" if torch.mps.is_available() else "cpu")

# Initialize the Feature Extractor (Encoder)
# We use a Multi-Kernel CNN to capture n-gram patterns of lengths 3, 4, and 5
encoder = TextCNNEncoder(
    vocab_size=len(vocab),  # Determined by the tokenizer in Section 6
    emb_dim=128,  # Dimensionality of the dense word vectors
    num_filters=128,  # Number of features to extract per kernel size
    kernel_sizes=(3, 4, 5),  # Window sizes: Tri-grams, 4-grams, 5-grams
    out_dim=128,  # Final embedding size (compact semantic vector)
    pad_idx=pad_id,  # Index to ignore during embedding lookup (zero gradient)
    dropout=0.4,  # Regularization to prevent overfitting on specific phrases
)

# Wrap the encoder in the Siamese architecture for metric learning
# scale=10.0 expands the cosine range [-1, 1] to [-10, 10] for sharper probability gradients
model = SiameseCNN(encoder).to(device)

print(f"Model initialized on: {device}")

Model initialized on: mps


### 4.5 Train the CNN

In [112]:
# 1. New Criterion for Metric Learning
# p=2 indicates Euclidean distance. With your L2-normalized CNN,
# this focuses the learning on the angular separation between users.
criterion = nn.TripletMarginLoss(margin=1.0, p=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)


@torch.no_grad()
def eval_triplet_acc(model, loader):
    model.eval()
    correct = 0
    total = 0

    # Unpacking the three context-aware tensors
    for _, _, _, u_tensor, v_pos_tensor, v_neg_tensor, _, _, _ in loader:
        u_tensor = u_tensor.to(device)
        v_pos_tensor = v_pos_tensor.to(device)
        v_neg_tensor = v_neg_tensor.to(device)

        # Get embeddings from SiameseCNN
        emb_u, emb_v_pos, emb_v_neg = model(u_tensor, v_pos_tensor, v_neg_tensor)

        # Calculate Euclidean distances: d(anchor, positive) vs d(anchor, negative)
        dist_pos = torch.norm(emb_u - emb_v_pos, p=2, dim=1)
        dist_neg = torch.norm(emb_u - emb_v_neg, p=2, dim=1)

        # A successful "ranking" occurs when the positive is closer than the hard negative
        correct += (dist_pos < dist_neg).sum().item()
        total += u_tensor.size(0)

    return correct / total if total > 0 else 0.0


def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0

    for _, _, _, u_tensor, v_pos_tensor, v_neg_tensor, _, _, _ in loader:
        u_tensor = u_tensor.to(device)
        v_pos_tensor = v_pos_tensor.to(device)
        v_neg_tensor = v_neg_tensor.to(device)

        optimizer.zero_grad(set_to_none=True)

        # Forward pass through shared Siamese weights
        emb_u, emb_v_pos, emb_v_neg = model(u_tensor, v_pos_tensor, v_neg_tensor)

        # The loss pulls u closer to v_pos and pushes it away from v_neg
        loss = criterion(emb_u, emb_v_pos, emb_v_neg)

        loss.backward()
        # Gradient clipping prevents the "exploding gradient" problem in deep CNNs
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * u_tensor.size(0)

    return total_loss / len(loader.dataset)


# --- Run Loop ---
EPOCHS = 5
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, train_loader)

    # Accuracy is the % of triplets correctly ranked
    test_acc = eval_triplet_acc(model, test_loader)
    train_acc = eval_triplet_acc(model, train_loader)

    print(
        f"Epoch {epoch:02d} | Loss: {loss:.4f} | Train Acc: {train_acc:.2%} | Test Acc: {test_acc:.2%}"
    )

KeyboardInterrupt: 

### 4.6 Export the User "Galaxy" (Embeddings)

In [ ]:
@torch.no_grad()
def generate_user_embeddings_batched(model, unique_user_df, batch_size=256):
    model.eval()
    user_ids = unique_user_df["author"].tolist()

    # 1. Pre-numericalize all texts using your existing 'encode' function
    # We pad them manually here to create a valid tensor
    all_seqs = [encode(t) for t in unique_user_df["body"]]
    max_len = max(len(s) for s in all_seqs)

    padded_seqs = torch.full((len(all_seqs), max_len), pad_id, dtype=torch.long)
    for i, s in enumerate(all_seqs):
        padded_seqs[i, : len(s)] = torch.tensor(s)

    # 2. Extract embeddings in batches using the GPU
    all_embs = []
    for i in range(0, len(padded_seqs), batch_size):
        batch = padded_seqs[i : i + batch_size].to(device)
        # Use ONLY the encoder; we don't need the Siamese wrapper for single users
        emb = model.encoder(batch)
        all_embs.append(emb.cpu().numpy())

    full_matrix = np.vstack(all_embs)
    return {u_id: vec for u_id, vec in zip(user_ids, full_matrix)}, full_matrix


# --- How to call it ---
# Create the unique user list from your test or train data
unique_users = (
    df_train.groupby("author")["body"]
    .apply(lambda s: " ".join(s.tail(10)))
    .reset_index()
)

user_vectors, embeddings_matrix = generate_user_embeddings_batched(model, unique_users)

### Create Graph Embeddings

In [ ]:
# ======================================================
# 5.X Train Node2Vec on training graph
# ======================================================

user_id_list = unique_users["author"].tolist()


# Build directed graph from training interactions
G = nx.DiGraph()

for u, v in pos_train[["u", "v"]].itertuples(index=False):
    G.add_edge(u, v)

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

# Train Node2Vec
node2vec = Node2Vec(
    G,
    dimensions=64,
    walk_length=30,
    num_walks=200,
    workers=4,
    p=1.0,
    q=1.0
)


n2v_model = Word2Vec(
    node2vec.walks, 
    vector_size=64, 
    window=10, 
    min_count=1, 
    batch_words=128
)

# Extract graph embeddings aligned with text embeddings
graph_dim = 64
graph_embeddings = {}

for user in user_id_list:
    if user in n2v_model.wv:
        graph_embeddings[user] = n2v_model.wv[user]
    else:
        graph_embeddings[user] = np.zeros(graph_dim)

Graph nodes: 999
Graph edges: 2722


Generating walks (CPU: 4): 100%|██████████| 50/50 [00:07<00:00,  6.57it/s]


In [ ]:
# Pick a random user from your list
sample_user = user_id_list[0]
print(f"Embedding for {sample_user}:\n", graph_embeddings[sample_user])

Embedding for -willis:
 [ 1.3569409e+00  5.1061481e-01  3.3593676e+00  3.8172561e-01
 -1.5182751e+00 -1.6614985e+00  1.5542983e+00  1.1570252e+00
 -9.9193484e-01  8.2261968e-01  1.3452015e+00 -1.2141443e+00
 -1.9324476e+00  2.1132667e-01  1.5083458e-01 -1.7133870e+00
  2.1686175e+00 -1.9132451e+00 -1.0368428e-01  1.0381227e+00
  1.9370266e+00 -1.5304898e+00 -1.6746415e+00 -1.6731048e+00
  2.7124861e-01  3.3236971e-01 -8.3264452e-01  1.0587667e+00
  1.8406663e+00 -1.5380386e+00  3.2741520e+00  1.6984259e+00
 -3.5302214e-02 -1.3559563e+00 -5.7383639e-01 -1.4879805e+00
 -7.1603477e-01 -1.1729325e-01 -1.3451487e-01  3.5458168e-01
  5.9422398e-01  6.1948228e-01  6.0629521e-02  1.5933852e-01
  1.1148442e+00  1.0900948e-03  1.3338758e+00 -6.8699640e-01
 -2.0027280e+00  9.6567255e-01  3.3169887e+00  2.8499961e+00
 -9.9171430e-01 -1.7984269e+00  5.9475327e-01  2.4613571e+00
  2.5571389e+00 -2.3274826e-01  3.2604115e+00  1.9392996e-01
  4.0501276e-01 -1.5850651e+00 -4.1474172e-01 -1.3047391e-01]

### 5.2 Set up a Vector Search Index

In [ ]:
# # 1. Prepare data (FAISS requires float32)
# embeddings_matrix = embeddings_matrix.astype("float32")
# user_id_list = unique_users["author"].tolist()

# # 2. Build the Index (IndexFlatIP = Inner Product)
# # Since your CNN uses F.normalize, Inner Product is equivalent to Cosine Similarity.
# d = embeddings_matrix.shape[1]
# index = faiss.IndexFlatIP(d)
# index.add(embeddings_matrix)

# # 3. Create a lookup dictionary for user_vectors if you haven't already
# # This allows O(1) retrieval of a vector by username
# user_vectors = {u_id: embeddings_matrix[i] for i, u_id in enumerate(user_id_list)}

# print(f"FAISS index built with {index.ntotal} users.")

In [ ]:
# ======================================================
# 5.2 Combine Text + Graph Embeddings and Build FAISS
# ======================================================
user_id_list = unique_users["author"].tolist()

combined_vectors = {}

for user in user_id_list:
    text_vec = user_vectors[user]
    graph_vec = graph_embeddings[user]
    combined = np.concatenate([text_vec, graph_vec])
    combined = combined / np.linalg.norm(combined)

    combined_vectors[user] = combined

# Build matrix
combined_matrix = np.vstack([
    combined_vectors[u] for u in user_id_list
]).astype("float32")

d = combined_matrix.shape[1]
index = faiss.IndexFlatIP(d)
index.add(combined_matrix)

# Update user_vectors to combined
user_vectors = {
    u: combined_matrix[i]
    for i, u in enumerate(user_id_list)
}

print("Hybrid FAISS index built with", index.ntotal, "users.")

Hybrid FAISS index built with 1321 users.


/var/folders/0x/mmpw_pbd0zl7qptfw7jdkhg40000gn/T/ipykernel_19619/3406313152.py:13: RuntimeWarning: invalid value encountered in divide
  combined = combined / np.linalg.norm(combined)


### 5.3 Implement the Recommendation Function

In [ ]:
train_edges = defaultdict(set)

for u, v in pos_train[["u", "v"]].itertuples(index=False):
    train_edges[u].add(v)   # only outgoing
    
def recommend_connections(query_user_id, k=10):
    """
    Returns top-k recommended users for a given query user.

    Excludes:
    - The query user itself
    - Users already connected in the training graph
    - Users without embeddings (implicitly handled via FAISS index)

    Expands search space if necessary.
    """

    # 1. Ensure query user has embedding
    if query_user_id not in user_vectors:
        return []

    # 2. Users already connected in training (undirected)
    existing_connections = train_edges.get(query_user_id, set())

    # 3. Prepare FAISS query
    query_vec = user_vectors[query_user_id].reshape(1, -1).astype("float32")

    # 4. We search progressively larger candidate pools
    max_candidates = len(user_id_list)
    search_k = min(max(k + 10, 50), max_candidates)  # reasonable initial pool

    # Perform FAISS search once per expansion
    while True:
        distances, indices = index.search(query_vec, search_k)

        results = []

        for idx in indices[0]:
            candidate = user_id_list[idx]

            # Exclude self
            if candidate == query_user_id:
                continue

            # Exclude train neighbors
            if candidate in existing_connections:
                continue

            results.append(candidate)

            if len(results) == k:
                return results

        # If we exhausted candidate pool, stop
        if search_k >= max_candidates:
            return results[:k]

        # Otherwise expand search space
        search_k = min(search_k * 2, max_candidates)

In [ ]:
def test_recommendation(user_id):
    if user_id not in user_vectors:
        return "User not found"

    recs = recommend_connections(user_id, k=3)

    print(
        f"Target User ({user_id}) history sample: {unique_users[unique_users['author'] == user_id]['body'].values[0][:100]}..."
    )
    print("-" * 30)
    for i, rec_id in enumerate(recs):
        text = unique_users[unique_users["author"] == rec_id]["body"].values[0][:100]
        print(f"Rec {i + 1}: {rec_id} | Text: {text}...")


# Example call
test_recommendation(user_id_list[15])

Target User (ANEPICLIE) history sample: In my opinion, to censor speech is to inhibit expression. To censor speech is to give precedent to c...
------------------------------
Rec 1: zombient | Text: I do not think I'm better off by having a third of my paycheck taken from me....
Rec 2: zombient | Text: I do not think I'm better off by having a third of my paycheck taken from me....
Rec 3: zombient | Text: I do not think I'm better off by having a third of my paycheck taken from me....


# 6. Evaluation

### 6.1 Build Ground Truth

In [ ]:
# 1. Build neighbor dictionaries
train_neighbors = pos_train.groupby("u")["v"].apply(set).to_dict()
test_neighbors = pos_test.groupby("u")["v"].apply(set).to_dict()

# 2. Users that exist in embedding index
embedded_users = set(user_vectors.keys())

ground_truth = {}

for u in test_neighbors:
    # Skip users without embeddings (cannot generate recommendations)
    if u not in embedded_users:
        continue

    train_set = train_neighbors.get(u, set())

    # Remove already seen interactions (only new links)
    new_interactions = test_neighbors[u] - train_set

    # Keep only targets that also have embeddings
    new_interactions = {v for v in new_interactions if v in embedded_users}

    # Only keep users with at least one evaluable target
    if len(new_interactions) > 0:
        ground_truth[u] = new_interactions


In [ ]:
# Number of valid future targets per user
gt_sizes = {u: len(vs) for u, vs in ground_truth.items()}

# Convert to DataFrame for easier inspection
gt_df = pd.DataFrame.from_dict(gt_sizes, orient="index", columns=["n_targets"])
gt_df.index.name = "user"

gt_df.head()


,n_targets
user,
294116002,1
A_Soporific,7
Amunium,1
Buster01,1
ByronicAsian,1


### 6.2 Evaluate Precision@K

In [ ]:
def evaluate_precision_at_k(model_recommend_fn, ground_truth, k=10):
    """
    model_recommend_fn: function(user_id, k) -> list of recommended users
    ground_truth: dict {u: set(valid target users)}
    """
    total_hits = 0
    total_users = 0

    for u, true_targets in ground_truth.items():
        recs = model_recommend_fn(u, k=k)

        # Safety: ensure only evaluable users are recommended
        recs = [r for r in recs if r in user_vectors]

        hits = len(set(recs) & true_targets)

        total_hits += hits
        total_users += 1

    if total_users == 0:
        return 0.0

    precision = total_hits / (k * total_users)
    return precision

In [ ]:
precision_cnn = evaluate_precision_at_k(recommend_connections, ground_truth, k=10)

print("CNN Precision@5:", precision_cnn)

CNN Precision@5: 0.0011904761904761906


### 6.3 Evaluate Recall@K

In [ ]:
def evaluate_recall_at_k(model_recommend_fn, ground_truth, k=10):
    """
    model_recommend_fn: function(user_id, k) -> list of recommended users
    ground_truth: dict {u: set(valid target users)}
    """

    total_recall = 0.0
    total_users = 0

    for u, true_targets in ground_truth.items():
        recs = model_recommend_fn(u, k=k)

        # Safety: ensure candidate universe consistency
        recs = [r for r in recs if r in user_vectors]

        hits = len(set(recs) & true_targets)

        recall_u = hits / len(true_targets)

        total_recall += recall_u
        total_users += 1

    if total_users == 0:
        return 0.0

    return total_recall / total_users

In [ ]:
recall_cnn = evaluate_recall_at_k(
    recommend_connections,
    ground_truth,
    k=10
)

print("CNN Recall@10:", recall_cnn)

CNN Recall@10: 0.011904761904761904


### 6.4 Evaluate nDCG@K

In [ ]:
def evaluate_ndcg_at_k(model_recommend_fn, ground_truth, k=10):
    """
    model_recommend_fn: function(user_id, k) -> ranked list
    ground_truth: dict {u: set(valid targets)}
    """
    
    total_ndcg = 0.0
    total_users = 0

    for u, true_targets in ground_truth.items():
        recs = model_recommend_fn(u, k=k)

        # Compute DCG
        dcg = 0.0
        for rank, candidate in enumerate(recs, start=1):
            if candidate in true_targets:
                dcg += 1.0 / np.log2(rank + 1)

        # Compute IDCG
        ideal_hits = min(len(true_targets), k)
        idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))

        if idcg == 0:
            continue  # skip users with no valid ground truth (safety)

        ndcg_u = dcg / idcg

        total_ndcg += ndcg_u
        total_users += 1

    if total_users == 0:
        return 0.0

    return total_ndcg / total_users

In [ ]:
ndcg_cnn = evaluate_ndcg_at_k(
    recommend_connections,
    ground_truth,
    k=10
)

print("CNN nDCG@10:", ndcg_cnn)

CNN nDCG@10: 0.0035836904245712046
